# Verificador de Inconsistências de Mensuradores — RPS Sesc SP

**Referência normativa:** Guia do RPS para o DR SP, Versão 1.0 – setembro de 2024, válido a partir de 2025.

---

## Estrutura hierárquica do RPS

```
Programa
  └── Atividade
        └── Subatividade  [campo: subatividade]
              └── Serviço [campo: servico]  ← define os mensuradores
```

---

## Fonte de dados

**Lakehouse:** `RPS` — tabela `siplan_lancamento_mensuradores`
Alimentada pelo Dataflow Gen2 `apontamentos_inconsistencias` via ODBC → HIVE.

Cada linha registra **um valor numérico** (geralmente pessoas) dentro de uma cadeia hierárquica,
qualificado por categoria de público e modalidade de acesso.

| Campo | Descrição |
|---|---|
| `atividade_id` | ID da ação programática |
| `sessao_id` | ID de cada sessão (data/horário) de uma mesma ação |
| `uo` | UO da atividade - número |
| `unidade` | Unidade da atividade - nome |
| `atividade_nome` | Nome da atividade |
| `areaprogramatica` | Área programática |
| `programa` | Programa RPS |
| `subatividade` | **Subatividade** (ex: Ações Formativas, Competições físico-esportivas…) |
| `servico` | **Serviço** (Oficina, Curso, Palestra…) — define os mensuradores |
| `data_sessao` | Data da sessão |
| `local` | Local de realização |
| `grupo_nome` | Tipo de mensurador (ex: `Presenças`, `Inscritos no dia`, `Partidas / Provas`, `Procedência dos inscritos`). ⚠️ Pode conter hífens — tratar cada parte como entidade distinta. |
| `variavel_nome` | Qualificação do público: `Pleno - Titular`, `Pleno - Dependente`, `MIS e Atividade`, `Não identificado` |
| `tipo_variavel_nome` | Modalidade de acesso: `Pago`, `PCG`, `Gratuito` |
| `valor_mensurador` | Valor numérico lançado |
| `lancamento_status` | Estado do lançamento: lançado / aprovado |

> **Nota:** as comparações somam todos os valores da sessão ou da atividade, sem desdobrar por `variavel_nome` ou `tipo_variavel_nome`.

---

## Regras de consistência implementadas

### [1] Oficina *(Subatividade: Ações Formativas)*
> Ação que tem **início, meio e fim na mesma sessão**, podendo ser ofertada várias vezes com o mesmo conteúdo.

**Regra:** Para cada sessão, apontar quando:
```
Σ Presenças  >  Σ Inscritos no dia
```
*Numa sessão única não podem comparecer mais pessoas do que as inscritas para aquela data.*

---

### [2] Curso *(Subatividade: Ações Formativas)*
> Ação com **encontros sequenciais** em datas diferentes, com o mesmo público durante todos os encontros.
> Mensuradores adicionais: `Concluintes` e `Evasões no dia`.

**Regra:** Para cada atividade (acumulado de todas as sessões), apontar quando:
```
Σ Presenças  <  Σ Inscritos no dia
```
*Ao longo de múltiplas sessões, o acumulado de presenças deve superar o total de inscrições, pois cada inscrito contribui com presenças em vários encontros.*

---

### [3] Curso — Inscritos sem nenhuma presença *(Subatividade: Ações Formativas)*

**Regra:** Para cada atividade (acumulado de todas as sessões), apontar quando:
```
Σ Inscritos > 0  e  Σ Presenças = 0
```
*Existindo inscritos, ao menos uma presença deve ter sido registrada em alguma sessão do curso.*

---

### [4] Competições Físico-Esportivas — Inscritos sem presença

**Regra:** Para cada atividade (acumulado de todas as sessões), apontar quando:
```
Σ Inscritos > 0  e  Σ Presenças = 0
```
*Existindo inscritos, ao menos uma presença deve ter sido registrada em alguma sessão da competição.*

---

### [5] Competições Físico-Esportivas — Partidas/Provas sem valor

**Regra:** Para cada sessão, apontar quando a soma de todos os campos do mensurador `Partidas / Provas` for:
```
Σ Partidas / Provas  =  0  (ou sem registro)
```
*Toda competição registrada deve indicar quantas partidas ou provas foram realizadas.*

---

### [6] Passeios — Presenças > Inscritos

**Regra:** Para cada sessão, apontar quando:
```
Σ Presenças  >  Σ Inscritos no dia
```
*Passeios requerem inscrição prévia; o número de participantes não pode superar o de inscritos.*

---

### [7] Viagens e Passeios — Procedência dos inscritos ≠ Total de inscritos

**Regra:** Para cada atividade (acumulado de todas as sessões), apontar quando:
```
Σ Procedência dos inscritos  ≠  Σ Inscritos no dia
```
*A soma das procedências declaradas deve ser igual ao total de inscritos. Exportar as colunas `soma_procedencia` e `soma_inscritos` na linha da atividade.*

---

### [8] Oficinas e Passeios — Inscritos sem presença

**Regra:** Para cada sessão, apontar quando:
```
Σ Inscritos > 0  e  Σ Presenças = 0
```
*Existindo inscritos, ao menos uma presença deve ter sido registrada na sessão.*

### [9] Sessões não aprovadas
**Regra:** Apontar todas as sessões que não estejam com lancamento_status = "APROVADO"

### [10] Intervenção urbana com valor ≠ 1
**Regras:**
- **Artes Visuais:** soma por `atividade_id × ano` deve ser igual a 1.
- **Demais áreas programáticas:** soma por sessão deve ser igual a 1.

## Configuração

In [2]:
import pandas as pd

# ── Parâmetros ────────────────────────────────────────────────────────────────
# Fonte: Lakehouse RPS no workspace "Dados Estatísticos SescSP"
# Alimentado pelo Dataflow Gen2 "apontamentos_inconsistencias" via ODBC → HIVE
LAKEHOUSE            = "RPS"
TABELA               = "siplan_lancamento_mensuradores"
SERVICOS_ATIVOS      = ["Oficina", "Curso", "Intervenção urbana"]
SUBATIVIDADES_ATIVAS = ["Competições físico-esportivas", "Passeios", "Viagens"]

# Mapeamento de nomes reais das colunas no HIVE → semântica do RPS
# (o modelo semântico renomeia algumas colunas; aqui usamos os nomes do HIVE)
#   areaprogramatica  = areaprogramatica_nome no modelo semântico
#   programa          = programa_desc
#   subatividade      = modalidade_desc
#   servico           = realizacao_desc

# Leitura dos Dados

In [3]:
print(f"Lendo dados do Lakehouse {LAKEHOUSE}...")

_servicos = ", ".join(f"'{s}'" for s in SERVICOS_ATIVOS)

# Carregar tabela e remover prefixo 'slm.' gravado pelo Dataflow
# Nota: 'spark' é injetado automaticamente pelo runtime do Fabric — aviso do IDE local pode ser ignorado
df_raw = spark.table(f"{LAKEHOUSE}.{TABELA}")  # noqa: F821
df_raw = df_raw.toDF(*[c.replace("slm.", "") for c in df_raw.columns])
df_raw.createOrReplaceTempView("lancamentos")

df = spark.sql(f"""
    SELECT
        uo, unidade, atividade_id, sessao_id, atividade_nome,
        areaprogramatica, programa, subatividade,
        TRIM(servico)            AS servico,
        data_sessao, local, lancamento_status,
        TRIM(grupo_nome)         AS grupo_nome,
        TRIM(variavel_nome)      AS variavel_nome,
        TRIM(tipo_variavel_nome) AS tipo_variavel_nome,
        valor_mensurador
    FROM lancamentos
    WHERE TRIM(servico) IN ({_servicos})
       OR TRIM(subatividade) IN ({", ".join(f"'{s}'" for s in SUBATIVIDADES_ATIVAS)})
""").toPandas()

print(f"Registros carregados: {len(df):>10,}")

Lendo dados do Lakehouse RPS...


NameError: name 'spark' is not defined

## Pré-processamento

In [3]:
# ── grupo_nome pode conter hífens indicando subgrupos ────────────────────────
# Exemplo: "Inscritos no dia - PCG" → grupo_principal="Inscritos no dia", grupo_sufixo="PCG"
# Tratar cada parte como entidade distinta (conforme orientação do RPS)
split = df["grupo_nome"].str.split(" - ", n=1, expand=True)
df["grupo_principal"] = split[0].str.strip()
df["grupo_sufixo"]    = split[1].str.strip() if 1 in split.columns else None

# ── Garantir valor numérico ───────────────────────────────────────────────────
df["valor_mensurador"] = pd.to_numeric(df["valor_mensurador"], errors="coerce").fillna(0)

print("Distribuição de grupo_principal nos dados filtrados:")
print(df["grupo_principal"].value_counts().to_string())

StatementMeta(, 09fc5be8-2f5d-44dc-bc09-fa31336f4ae8, 5, Finished, Available, Finished, False)

Distribuição de grupo_principal nos dados filtrados:
grupo_principal
Presenças                    104929
Inscritos no dia             104508
Pessoas atendidas            102882
Concluintes                   64869
Evasões no dia                64853
null                          11501
Procedência dos inscritos       909
Competição                      304


## Funções auxiliares: pivot e sem presença

In [4]:
GRUPOS_MENSURADOS = ["Presenças", "Inscritos no dia"]

def pivotar_mensuradores(df_servico: pd.DataFrame, chave: list[str]) -> pd.DataFrame:
    """
    Agrega e pivota o DataFrame para colocar 'Presenças' e 'Inscritos no dia'
    como colunas, permitindo a comparação direta.

    Parâmetros
    ----------
    df_servico : DataFrame filtrado pelo serviço (Oficina ou Curso)
    chave      : colunas de agrupamento (define o nível de comparação)

    Retorna
    -------
    DataFrame com colunas: <chave> + ['presencas', 'inscritos']
    """
    pivot = (
        df_servico[df_servico["grupo_principal"].isin(GRUPOS_MENSURADOS)]
        .groupby(chave + ["grupo_principal"])["valor_mensurador"]
        .sum()
        .unstack(fill_value=0)
        .reset_index()
    )
    pivot.columns.name = None

    # Garantir colunas mesmo que não haja registros de um dos tipos
    for col in GRUPOS_MENSURADOS:
        if col not in pivot.columns:
            pivot[col] = 0

    return pivot.rename(columns={
        "Presenças"       : "presencas",
        "Inscritos no dia": "inscritos",
    })


def sem_presenca(pivot: pd.DataFrame, servico: str, contexto: str) -> pd.DataFrame:
    """Filtra linhas com inscritos > 0 e presencas == 0 (Regra 3)."""
    incons = pivot[
        (pivot["inscritos"] > 0) & (pivot["presencas"] == 0)
    ].copy()
    incons["servico"]   = servico
    incons["regra"]     = "Inscritos sem nenhuma presença"
    incons["descricao"] = (
        f"Há inscritos nesta categoria mas nenhuma presença registrada {contexto}."
    )
    incons["diferenca"] = incons["inscritos"]
    return incons

StatementMeta(, 09fc5be8-2f5d-44dc-bc09-fa31336f4ae8, 6, Finished, Available, Finished, False)

## Regra 1 — Oficinas
**Presenças ≤ Inscritos no dia** (por sessão × categoria)

In [5]:
df_oficinas = df[df["servico"] == "Oficina"]

CHAVE_OFICINA = [
    "atividade_id", "uo", "unidade", "sessao_id", "atividade_nome",
    "areaprogramatica", "programa", "subatividade",
    "data_sessao", "local",
]

pivot_oficinas = pivotar_mensuradores(df_oficinas, CHAVE_OFICINA)

# Regra 1 — Presenças > Inscritos
incons_oficinas = pivot_oficinas[
    pivot_oficinas["presencas"] > pivot_oficinas["inscritos"]
].copy()
incons_oficinas["servico"]   = "Oficina"
incons_oficinas["regra"]     = "Presenças > Inscritos no dia (por sessão)"
incons_oficinas["descricao"] = (
    "Oficina é sessão única: não podem comparecer mais pessoas "
    "do que as inscritas para aquela data."
)
incons_oficinas["diferenca"] = incons_oficinas["presencas"] - incons_oficinas["inscritos"]

# Regra 8 — Inscritos sem nenhuma presença
incons_oficinas_sem_presenca = sem_presenca(pivot_oficinas, "Oficina", "na sessão")

print(f"[OFICINA] Sessões analisadas    : {len(pivot_oficinas):>6,}")
print(f"[OFICINA] Presenças > Inscritos : {len(incons_oficinas):>6,}")
print(f"[OFICINA] Inscritos sem presença: {len(incons_oficinas_sem_presenca):>6,}")

StatementMeta(, 09fc5be8-2f5d-44dc-bc09-fa31336f4ae8, 7, Finished, Available, Finished, False)

[OFICINA] Sessões analisadas    : 36,191
[OFICINA] Inconsistências       :    479


## Regras 2 e 3 — Cursos
**Total Presenças ≥ Total Inscritos no dia** (por atividade, acumulado de todas as sessões)

In [6]:
df_cursos = df[df["servico"] == "Curso"]

# Nível de comparação: atividade completa (acumula todas as sessões)
CHAVE_CURSO = [
    "atividade_id", "uo", "unidade", "atividade_nome",
    "areaprogramatica", "programa", "subatividade",
    "local",
]

pivot_cursos = pivotar_mensuradores(df_cursos, CHAVE_CURSO)

# Regra 2 — Total Presenças < Total Inscritos
incons_cursos = pivot_cursos[
    pivot_cursos["presencas"] < pivot_cursos["inscritos"]
].copy()
incons_cursos["servico"]   = "Curso"
incons_cursos["regra"]     = "Total Presenças < Total Inscritos no dia (acumulado)"
incons_cursos["descricao"] = (
    "Curso tem múltiplas sessões: o acumulado de presenças deve superar "
    "o total de inscrições, pois cada inscrito contribui com presenças "
    "em vários encontros."
)
incons_cursos["diferenca"] = incons_cursos["inscritos"] - incons_cursos["presencas"]

# Regra 3 — Inscritos sem nenhuma presença
incons_cursos_sem_presenca = sem_presenca(
    pivot_cursos, "Curso", "em qualquer sessão do curso"
)

print(f"[CURSO] Atividades analisadas  : {len(pivot_cursos):>6,}")
print(f"[CURSO] Presenças < Inscritos  : {len(incons_cursos):>6,}")
print(f"[CURSO] Inscritos sem presença : {len(incons_cursos_sem_presenca):>6,}")

StatementMeta(, 09fc5be8-2f5d-44dc-bc09-fa31336f4ae8, 8, Finished, Available, Finished, False)

[CURSO]   Atividades analisadas : 13,426
[CURSO]   Inconsistências       :  1,438


## Regra 4 — Competições: Inscritos sem presença
Por atividade (acumulado de todas as sessões)

In [7]:
df_comp = df[df["subatividade"].str.strip() == "Competições físico-esportivas"]

# Nível de comparação: atividade completa (acumula todas as sessões)
CHAVE_COMP_ACUM = [
    "atividade_id", "uo", "unidade", "atividade_nome",
    "areaprogramatica", "programa", "subatividade",
    "local",
]

pivot_comp = pivotar_mensuradores(df_comp, CHAVE_COMP_ACUM)

# Regra 4 — Inscritos sem nenhuma presença
incons_comp_sem_presenca = sem_presenca(
    pivot_comp, "Competição Físico-Esportiva", "na competição"
)

print(f"[COMPETIÇÃO] Atividades analisadas  : {len(pivot_comp):>6,}")
print(f"[COMPETIÇÃO] Inscritos sem presença : {len(incons_comp_sem_presenca):>6,}")

StatementMeta(, 09fc5be8-2f5d-44dc-bc09-fa31336f4ae8, 9, Finished, Available, Finished, False)

[CURSO - sem presença] Inconsistências:  1,007


## Regra 5 — Competições: Partidas/Provas sem valor
Apontar quando a soma de todos os campos do mensurador `Partidas / Provas` for zero ou sem registro.

In [10]:
CHAVE_SESSAO_COMP = [
    "uo", "unidade",
    "atividade_id", "sessao_id", "atividade_nome",
    "areaprogramatica", "programa", "subatividade",
    "data_sessao", "local",
]

# Chave no nível da atividade (sem sessao_id, data_sessao e local)
CHAVE_ATIV_COMP = [
    "uo", "unidade",
    "atividade_id", "atividade_nome",
    "areaprogramatica", "programa", "subatividade",
]

# Filtra somente as linhas onde variavel_nome == "Partidas / Provas".
# Em competições, grupo_nome="Competição" e variavel_nome identifica o tipo.
df_partidas = df_comp[df_comp["variavel_nome"] == "Partidas / Provas"]

# Soma por atividade_id — se a atividade tem ≥1 partida lançada, está ok.
soma_partidas = (
    df_partidas
    .groupby(CHAVE_ATIV_COMP)["valor_mensurador"]
    .sum()
    .reset_index()
    .rename(columns={"valor_mensurador": "soma_partidas"})
)

# Todas as atividades de competição — as sem nenhum registro ficam com soma 0 via fillna
todas_atividades = df_comp[CHAVE_ATIV_COMP].drop_duplicates()

comp_partidas = todas_atividades.merge(soma_partidas, on=CHAVE_ATIV_COMP, how="left")
comp_partidas["soma_partidas"] = comp_partidas["soma_partidas"].fillna(0)

# Apontar atividades onde a soma de "Partidas / Provas" é 0 (valor zero ou sem registro)
incons_partidas = comp_partidas[comp_partidas["soma_partidas"] == 0].copy()

incons_partidas["servico"]         = "Competição Físico-Esportiva"
incons_partidas["regra"]           = "Partidas / Provas sem valor"
incons_partidas["descricao"]       = (
    "Soma de 'Partidas / Provas' na atividade é zero ou sem registro."
)
incons_partidas["partidas_provas"]  = 0
incons_partidas["diferenca"]        = None

print(f"[COMPETIÇÃO - Partidas/Provas] Atividades analisadas: {len(comp_partidas):>6,}")
print(f"[COMPETIÇÃO - Partidas/Provas] Soma = 0 ou s/ reg   : {len(incons_partidas):>6,}")
display(incons_partidas[CHAVE_ATIV_COMP + ["soma_partidas"]])

StatementMeta(, 09fc5be8-2f5d-44dc-bc09-fa31336f4ae8, 12, Finished, Available, Finished, False)

[COMPETIÇÃO - Partidas/Provas] Com valor zero  :      0
[COMPETIÇÃO - Partidas/Provas] Sem registro    :    227
[COMPETIÇÃO - Partidas/Provas] Total           :    227


SynapseWidget(Synapse.DataFrame, 146affac-6913-4dec-8ed1-b40e66ce67b4)

## Regras 6 e 8 — Passeios
Presenças > Inscritos e Inscritos sem presença — por sessão

In [ ]:
df_passeios = df[df["subatividade"].str.strip() == "Passeios"]

CHAVE_PASSEIO = [
    "atividade_id", "uo", "unidade", "sessao_id", "atividade_nome",
    "areaprogramatica", "programa", "subatividade",
    "data_sessao", "local",
]

pivot_passeios = pivotar_mensuradores(df_passeios, CHAVE_PASSEIO)

# Regra 6 — Presenças > Inscritos
incons_passeios = pivot_passeios[
    pivot_passeios["presencas"] > pivot_passeios["inscritos"]
].copy()
incons_passeios["servico"]   = "Passeio"
incons_passeios["regra"]     = "Presenças > Inscritos (por sessão)"
incons_passeios["descricao"] = "Número de presenças superior ao de inscritos no passeio."
incons_passeios["diferenca"] = incons_passeios["presencas"] - incons_passeios["inscritos"]

# Regra 8 — Inscritos sem nenhuma presença
incons_passeios_sem_presenca = sem_presenca(pivot_passeios, "Passeio", "no passeio")

print(f"[PASSEIO] Sessões analisadas    : {len(pivot_passeios):>6,}")
print(f"[PASSEIO] Presenças > Inscritos : {len(incons_passeios):>6,}")
print(f"[PASSEIO] Inscritos sem presença: {len(incons_passeios_sem_presenca):>6,}")

: 

## Regra 7 — Viagens e Passeios: Procedência dos inscritos
Apontar quando `Σ Procedência dos inscritos ≠ Σ Inscritos no dia` por atividade.

In [20]:
df_viagens = df[df["subatividade"].str.strip() == "Viagens"]

CHAVE_PROC = [
    "uo", "unidade",
    "atividade_id", "atividade_nome",
    "areaprogramatica", "programa", "subatividade", "local",
]

def verificar_procedencia(df_sub, servico):
    """Compara Σ Procedência dos inscritos com Σ Inscritos no dia por atividade."""
    soma_ins = (
        df_sub[df_sub["grupo_principal"] == "Inscritos no dia"]
        .groupby(CHAVE_PROC)["valor_mensurador"]
        .sum()
        .reset_index()
        .rename(columns={"valor_mensurador": "soma_inscritos"})
    )
    # Usa grupo_principal para capturar variantes como
    # "Procedência dos inscritos - Capital", "Procedência dos inscritos - Interior" etc.
    soma_proc = (
        df_sub[df_sub["grupo_principal"] == "Procedência dos inscritos"]
        .groupby(CHAVE_PROC)["valor_mensurador"]
        .sum()
        .reset_index()
        .rename(columns={"valor_mensurador": "soma_procedencia"})
    )
    comp = soma_ins.merge(soma_proc, on=CHAVE_PROC, how="inner")
    incons = comp[comp["soma_procedencia"] != comp["soma_inscritos"]].copy()
    incons["regra"]     = "Procedência dos inscritos ≠ Total de Inscritos"
    incons["descricao"] = (
        "A soma dos valores de 'Procedência dos inscritos' é diferente "
        "do total de inscritos."
    )
    incons["diferenca"] = incons["soma_procedencia"] - incons["soma_inscritos"]
    incons["servico"]   = servico
    return comp, incons

comp_proc_viagens,  incons_proc_viagens  = verificar_procedencia(df_viagens,  "Viagem")
comp_proc_passeios, incons_proc_passeios = verificar_procedencia(df_passeios, "Passeio")

print(f"[VIAGEM  - procedência] Atividades comparadas: {len(comp_proc_viagens):>6,}")
print(f"[VIAGEM  - procedência] Inconsistências      : {len(incons_proc_viagens):>6,}")
print(f"[PASSEIO - procedência] Atividades comparadas: {len(comp_proc_passeios):>6,}")
print(f"[PASSEIO - procedência] Inconsistências      : {len(incons_proc_passeios):>6,}")

StatementMeta(, 09fc5be8-2f5d-44dc-bc09-fa31336f4ae8, 22, Finished, Available, Finished, False)

NameError: name 'pivot_comp' is not defined

## Regra 9 — Sessões não aprovadas
Apontar todas as sessões que não estejam com lancamento_status = "APROVADO"

In [ ]:
STATUS_APROVADO = "APROVADO"

CHAVE_SESSAO_R9 = [
    "uo", "unidade",
    "atividade_id", "sessao_id", "atividade_nome",
    "areaprogramatica", "programa", "subatividade", "servico",
    "data_sessao", "local", "lancamento_status",
]

incons_nao_aprovadas = (
    df[df["lancamento_status"] != STATUS_APROVADO]
    [CHAVE_SESSAO_R9]
    .drop_duplicates()
    .assign(
        regra="Sessão não aprovada",
        descricao=lambda x: "Status: " + x["lancamento_status"].astype(str),
    )
)

print(f"Regra 9 — Sessões não aprovadas: {len(incons_nao_aprovadas):>6,}")
display(incons_nao_aprovadas)

## Regra 10 — Intervenção urbana com valor ≠ 1
- **Artes Visuais:** soma por `atividade_id × ano` deve ser igual a 1.
- **Demais áreas programáticas:** soma por sessão deve ser igual a 1.

In [ ]:
df_interv = df[df["servico"].str.strip() == "Intervenção urbana"].copy()
df_interv["ano"] = pd.to_datetime(df_interv["data_sessao"], errors="coerce").dt.year

# ── Sub-regra A: Artes Visuais — soma por atividade × ano ────────────────
CHAVE_ARTES = [
    "uo", "unidade",
    "atividade_id", "atividade_nome",
    "areaprogramatica", "programa", "subatividade", "servico", "ano",
]

incons_interv_artes = (
    df_interv[df_interv["areaprogramatica"] == "Artes Visuais"]
    .groupby(CHAVE_ARTES, as_index=False)["valor_mensurador"]
    .sum()
    .query("valor_mensurador != 1")
    .assign(
        regra="Intervenção urbana com valor ≠ 1",
        descricao=lambda x: (
            "Soma anual = " + x["valor_mensurador"].astype(str)
            + " (ano " + x["ano"].astype(str) + ")"
        ),
        diferenca=lambda x: x["valor_mensurador"] - 1,
    )
)

# ── Sub-regra B: demais áreas — verifica por sessão ──────────────────────
CHAVE_SESSAO_INTERV = [
    "uo", "unidade",
    "atividade_id", "sessao_id", "atividade_nome",
    "areaprogramatica", "programa", "subatividade", "servico",
    "data_sessao", "local",
]

incons_interv_outras = (
    df_interv[df_interv["areaprogramatica"] != "Artes Visuais"]
    .groupby(CHAVE_SESSAO_INTERV, as_index=False)["valor_mensurador"]
    .sum()
    .query("valor_mensurador != 1")
    .assign(
        regra="Intervenção urbana com valor ≠ 1",
        descricao=lambda x: "Valor na sessão = " + x["valor_mensurador"].astype(str),
        diferenca=lambda x: x["valor_mensurador"] - 1,
    )
)

incons_interv_urbana = pd.concat([incons_interv_artes, incons_interv_outras], ignore_index=True)

print(f"Regra 10 — Intervenção urbana (Artes Visuais, por ano): {len(incons_interv_artes):>6,}")
print(f"Regra 10 — Intervenção urbana (demais, por sessão)    : {len(incons_interv_outras):>6,}")
display(incons_interv_urbana)


## Consolidação e resultado final

In [19]:
COLUNAS_SAIDA = [
    "servico", "uo", "unidade",
    "atividade_id", "sessao_id", "atividade_nome",
    "areaprogramatica", "programa", "subatividade",
    "data_sessao", "local",
    "inscritos", "presencas", "partidas_provas",
    "soma_inscritos", "soma_procedencia",
    "diferenca",
    "primeira_data",
    "lancamento_status",
    "regra", "descricao",
]

# Cada dataframe é realinhado para COLUNAS_SAIDA; colunas ausentes ficam None
todos = [
    ("Regra 1 — Oficinas"           , incons_oficinas),
    ("Regra 2 — Cursos"             , incons_cursos),
    ("Regra 3 — Cursos sem pres."   , incons_cursos_sem_presenca),
    ("Regra 4 — Comp. sem pres."    , incons_comp_sem_presenca),
    ("Regra 5 — Partidas/Provas"    , incons_partidas),
    ("Regra 6 — Passeios"           , incons_passeios),
    ("Regra 7 — Procedência Viagem" , incons_proc_viagens),
    ("Regra 7 — Procedência Passeio", incons_proc_passeios),
    ("Regra 8 — Oficinas sem pres." , incons_oficinas_sem_presenca),
    ("Regra 8 — Passeios sem pres." , incons_passeios_sem_presenca),
    ("Regra 9 — Sessões não aprov."  , incons_nao_aprovadas),
    ("Regra 10 — Interv. urbana"      , incons_interv_urbana),
]

inconsistencias = (
    pd.concat([d for _, d in todos if len(d) > 0], ignore_index=True)
    .reindex(columns=COLUNAS_SAIDA)
    .sort_values(["servico", "atividade_nome"])
    .reset_index(drop=True)
)

# data_sessao como date (sem hora)
inconsistencias["data_sessao"] = (
    pd.to_datetime(inconsistencias["data_sessao"], errors="coerce").dt.date
)

# primeira_data = MIN(data_sessao) de todas as sessões da atividade no dataset
# descarta a coluna None criada pelo reindex antes de popular via merge
inconsistencias = inconsistencias.drop(columns=["primeira_data"], errors="ignore")

_min_data = (
    df.assign(data_sessao=pd.to_datetime(df["data_sessao"], errors="coerce").dt.date)
    .groupby("atividade_id")["data_sessao"]
    .min()
    .reset_index()
    .rename(columns={"data_sessao": "primeira_data"})
)
inconsistencias = inconsistencias.merge(_min_data, on="atividade_id", how="left")


# ── Diagnóstico: servico nulo ────────────────────────────────────
nulos = inconsistencias[inconsistencias["servico"].isna()]
print(f"Linhas com servico nulo: {len(nulos)}")
display(nulos[["atividade_id", "regra", "servico", "descricao"]])

# verificar em qual df de origem o registro existe
aid = 71000006911168
for label, d in todos:
    sub = d[d["atividade_id"] == aid] if "atividade_id" in d.columns else d.iloc[0:0]
    if len(sub):
        print(f"Encontrado em: {label}")
        display(sub)
# ── Resumo ────────────────────────────────────────────────────────────────────────
print("=" * 60)
print(f"  TOTAL DE INCONSISTÊNCIAS: {len(inconsistencias)}")
print("=" * 60)
for label, d in todos:
    print(f"  {label:<30}: {len(d)}")
print("=" * 60)

display(inconsistencias)

# ── Gravar no Lakehouse para consumo no Power BI ──────────────────────────
spark.createDataFrame(inconsistencias).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("RPS.inconsistencias_rps")  # noqa: F821
print("Tabela 'inconsistencias_rps' gravada no Lakehouse RPS.")


StatementMeta(, 09fc5be8-2f5d-44dc-bc09-fa31336f4ae8, 21, Finished, Available, Finished, False)

NameError: name 'incons_oficinas_sem_presenca' is not defined